# Multimodal Fusion: Cross-Modal Multi-Head Attention

This notebook implements **Cross-Modal Attention Fusion** for in-hospital mortality prediction.
- **Temporal Sequence Features**: $t \in \mathbb{R}^{64}$
- **Static Tabular Features**: $s \in \mathbb{R}^{64}$
- **Mechanism**: The modalities are stacked as tokens $[t; s] \in \mathbb{R}^{2 \times 64}$. Multi-head self/cross-attention allows each modality to attend to the other, followed by residual connections, LayerNorm, and a feedforward network.


## Step 1: Imports, Hyperparameters & Device Selection
Initialize libraries, set random seeds for reproducibility, and configure hardware acceleration (MPS / CUDA / CPU).


In [2]:
from pathlib import Path
import copy
import json
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

from sklearn.metrics import (
    average_precision_score, roc_auc_score,
    precision_recall_curve, accuracy_score,
    balanced_accuracy_score, precision_score,
    recall_score, f1_score, confusion_matrix
)

SEED = 42
BATCH_SIZE = 512
MAX_EPOCHS = 50
PATIENCE = 7
LR = 1e-3

torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

_start = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in [_start, *_start.parents] if (p / "checkpoints").exists() and (p / "processed").exists()), _start)
FUSION_DIR = PROJECT_ROOT / "checkpoints" / "fusion"
OUT_DIR = FUSION_DIR / "attention_baseline"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("Device:", device)
print("Output:", OUT_DIR.resolve())




Device: mps
Output: /Users/aryandwivedi/Desktop/Project Hail Minor/checkpoints/fusion/attention_baseline


## Step 2: Load Multimodal Embeddings & Validate Disjointness
Load separate 64-dimensional temporal and tabular embeddings from `fused_embeddings_{split}.npz` and verify patient disjointness.


In [3]:
def load_split(split):
    path = FUSION_DIR / f"fused_embeddings_{split}.npz"

    if not path.exists():
        raise FileNotFoundError(path)

    with np.load(path) as data:
        temporal = data["temporal_embeddings"].astype(np.float32)
        tabular = data["tabular_embeddings"].astype(np.float32)
        ids = data["patient_ids"].reshape(-1).astype(np.int64)
        labels = data["labels"].reshape(-1).astype(np.float32)

    assert temporal.shape == tabular.shape
    assert temporal.ndim == 2 and temporal.shape[1] == 64
    assert len(temporal) == len(ids) == len(labels)
    assert len(np.unique(ids)) == len(ids)
    assert np.isfinite(temporal).all()
    assert np.isfinite(tabular).all()
    assert np.isfinite(labels).all()
    assert set(np.unique(labels)).issubset({0.0, 1.0})

    return temporal, tabular, labels, ids


t_train, s_train, y_train, ids_train = load_split("train")
t_val, s_val, y_val, ids_val = load_split("val")
t_test, s_test, y_test, ids_test = load_split("test")

assert not (set(ids_train) & set(ids_val))
assert not (set(ids_train) & set(ids_test))
assert not (set(ids_val) & set(ids_test))

print("Train:", t_train.shape)
print("Validation:", t_val.shape)
print("Test:", t_test.shape)



Train: (153831, 64)
Validation: (24186, 64)
Test: (22809, 64)


## Step 3: Modality-Specific Feature Standardization
Fit separate scalers on temporal and tabular vectors using training data only to respect differing feature scales.


In [4]:
# Fit separate scalers on training embeddings only.
t_mean, t_std = t_train.mean(axis=0), t_train.std(axis=0)
s_mean, s_std = s_train.mean(axis=0), s_train.std(axis=0)

t_std[t_std < 1e-8] = 1.0
s_std[s_std < 1e-8] = 1.0

t_train = ((t_train - t_mean) / t_std).astype(np.float32)
t_val = ((t_val - t_mean) / t_std).astype(np.float32)
t_test = ((t_test - t_mean) / t_std).astype(np.float32)

s_train = ((s_train - s_mean) / s_std).astype(np.float32)
s_val = ((s_val - s_mean) / s_std).astype(np.float32)
s_test = ((s_test - s_mean) / s_std).astype(np.float32)

assert np.isfinite(t_train).all() and np.isfinite(s_train).all()
assert np.isfinite(t_val).all() and np.isfinite(s_val).all()
assert np.isfinite(t_test).all() and np.isfinite(s_test).all()

np.savez(
    OUT_DIR / "input_scaler.npz",
    temporal_mean=t_mean,
    temporal_std=t_std,
    tabular_mean=s_mean,
    tabular_std=s_std
)




## Step 4: PyTorch DataLoaders & Class Balancing
Create data loaders feeding paired `(temporal, tabular, label)` tensors and calculate the positive class weighting ratio.


In [5]:
def make_loader(t, s, y, shuffle):
    dataset = TensorDataset(
        torch.from_numpy(t),
        torch.from_numpy(s),
        torch.from_numpy(y.astype(np.float32))
    )
    return DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        num_workers=0
    )


train_loader = make_loader(t_train, s_train, y_train, True)
val_loader = make_loader(t_val, s_val, y_val, False)
test_loader = make_loader(t_test, s_test, y_test, False)




## Step 5: Attention Fusion Classifier Architecture
Define `AttentionFusionClassifier` with 4-head attention, LayerNorm, FFN, and linear classifier head. Initialize optimizer and evaluation helper.


In [6]:
class AttentionFusionClassifier(nn.Module):
    def __init__(self, embedding_dim=64, num_heads=4, dropout=0.25):
        super().__init__()
        self.attention = nn.MultiheadAttention(
            embed_dim=embedding_dim,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True
        )
        self.norm1 = nn.LayerNorm(embedding_dim)
        self.ffn = nn.Sequential(
            nn.Linear(embedding_dim, embedding_dim * 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(embedding_dim * 2, embedding_dim)
        )
        self.norm2 = nn.LayerNorm(embedding_dim)
        self.classifier = nn.Sequential(
            nn.Linear(embedding_dim, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1)
        )

    def forward(self, temporal, tabular):
        modalities = torch.stack([temporal, tabular], dim=1)
        attended, _ = self.attention(
            modalities, modalities, modalities,
            need_weights=False
        )
        x = self.norm1(modalities + attended)
        x = self.norm2(x + self.ffn(x))
        fused = x.mean(dim=1)
        logits = self.classifier(fused).squeeze(-1)
        return logits


model = AttentionFusionClassifier().to(device)

n_pos = int((y_train == 1).sum())
n_neg = int((y_train == 0).sum())

criterion = nn.BCEWithLogitsLoss(
    pos_weight=torch.tensor(
        n_neg / max(n_pos, 1),
        dtype=torch.float32,
        device=device
    )
)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LR,
    weight_decay=1e-4
)

print(f"Training positives: {n_pos:,}")
print(f"Training negatives: {n_neg:,}")


def predict_probabilities(loader):
    model.eval()
    labels, probabilities = [], []

    with torch.no_grad():
        for temporal, tabular, y in loader:
            temporal = temporal.to(device)
            tabular = tabular.to(device)

            probs = torch.sigmoid(model(temporal, tabular))
            labels.append(y.numpy())
            probabilities.append(probs.cpu().numpy())

    return np.concatenate(labels), np.concatenate(probabilities)




Training positives: 13,634
Training negatives: 140,197


## Step 6: Training Loop with Validation AUPRC Early Stopping
Optimize the network using Adam with gradient clipping. Checkpoint the model with the highest validation AUPRC.


In [ ]:
best_val_ap = -np.inf
best_state = None
wait = 0
history = []

for epoch in range(MAX_EPOCHS):
    model.train()
    total_loss = 0.0
    total_samples = 0

    for temporal, tabular, labels in train_loader:
        temporal = temporal.to(device)
        tabular = tabular.to(device)
        labels = labels.to(device)

        optimizer.zero_grad(set_to_none=True)
        logits = model(temporal, tabular)
        loss = criterion(logits, labels)

        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item() * len(labels)
        total_samples += len(labels)

    train_loss = total_loss / total_samples
    val_labels, val_probs = predict_probabilities(val_loader)

    val_ap = average_precision_score(val_labels, val_probs)
    val_auc = roc_auc_score(val_labels, val_probs)

    history.append({
        "epoch": epoch + 1,
        "train_loss": float(train_loss),
        "val_auprc": float(val_ap),
        "val_auroc": float(val_auc)
    })

    print(
        f"Epoch {epoch + 1:02d}/{MAX_EPOCHS} | "
        f"Train loss={train_loss:.4f} | "
        f"Val AUPRC={val_ap:.4f} | "
        f"Val AUROC={val_auc:.4f}"
    )

    if np.isfinite(val_ap) and val_ap > best_val_ap:
        best_val_ap = val_ap
        best_state = copy.deepcopy(model.state_dict())
        wait = 0
    else:
        wait += 1
        if wait >= PATIENCE:
            print("Early stopping.")
            break

if best_state is None:
    raise RuntimeError("No valid checkpoint selected.")

model.load_state_dict(best_state)

torch.save(
    model.state_dict(),
    OUT_DIR / "attention_classifier.pt"
)

with open(OUT_DIR / "training_history.json", "w") as f:
    json.dump(history, f, indent=2)

print(f"Best validation AUPRC: {best_val_ap:.4f}")



Epoch 01/50 | Train loss=0.7360 | Val AUPRC=0.5159 | Val AUROC=0.8766
Epoch 02/50 | Train loss=0.7045 | Val AUPRC=0.5166 | Val AUROC=0.8757
Epoch 03/50 | Train loss=0.6983 | Val AUPRC=0.5229 | Val AUROC=0.8784
Epoch 04/50 | Train loss=0.6925 | Val AUPRC=0.5192 | Val AUROC=0.8773
Epoch 05/50 | Train loss=0.6887 | Val AUPRC=0.5227 | Val AUROC=0.8768
Epoch 06/50 | Train loss=0.6853 | Val AUPRC=0.5223 | Val AUROC=0.8774
Epoch 07/50 | Train loss=0.6849 | Val AUPRC=0.5253 | Val AUROC=0.8784
Epoch 08/50 | Train loss=0.6805 | Val AUPRC=0.5186 | Val AUROC=0.8766
Epoch 09/50 | Train loss=0.6786 | Val AUPRC=0.5276 | Val AUROC=0.8788
Epoch 10/50 | Train loss=0.6770 | Val AUPRC=0.5270 | Val AUROC=0.8787
Epoch 11/50 | Train loss=0.6747 | Val AUPRC=0.5258 | Val AUROC=0.8792
Epoch 12/50 | Train loss=0.6731 | Val AUPRC=0.5251 | Val AUROC=0.8760
Epoch 13/50 | Train loss=0.6721 | Val AUPRC=0.5276 | Val AUROC=0.8775
Epoch 14/50 | Train loss=0.6714 | Val AUPRC=0.5254 | Val AUROC=0.8768


## Step 7: Decision Threshold Selection (Validation F1)
Determine the optimal probability threshold by evaluating F1 scores across the validation precision-recall curve.


In [ ]:
# Select the classification threshold using validation F1.
val_labels, val_probs = predict_probabilities(val_loader)
p_curve, r_curve, thresholds = precision_recall_curve(
    val_labels, val_probs
)

f1_curve = (
    2 * p_curve[:-1] * r_curve[:-1]
    / (p_curve[:-1] + r_curve[:-1] + 1e-12)
)

best_idx = int(np.argmax(f1_curve))
best_threshold = float(thresholds[best_idx])

print(f"Selected threshold: {best_threshold:.4f}")
print(f"Validation F1: {f1_curve[best_idx]:.4f}")



## Step 8: Held-Out Test Evaluation & Metrics Export
Evaluate the held-out test cohort once with the selected threshold and export `test_metrics.json`.


In [ ]:
# Evaluate the held-out test set with the selected threshold.
test_labels, test_probs = predict_probabilities(test_loader)
test_pred = (test_probs >= best_threshold).astype(int)

results = {
    "fusion": "attention",
    "classifier": "MLP",
    "threshold": best_threshold,
    "test_auroc": float(roc_auc_score(test_labels, test_probs)),
    "test_auprc": float(
        average_precision_score(test_labels, test_probs)
    ),
    "test_accuracy": float(
        accuracy_score(test_labels, test_pred)
    ),
    "test_balanced_accuracy": float(
        balanced_accuracy_score(test_labels, test_pred)
    ),
    "test_precision": float(
        precision_score(test_labels, test_pred, zero_division=0)
    ),
    "test_recall": float(
        recall_score(test_labels, test_pred, zero_division=0)
    ),
    "test_f1": float(
        f1_score(test_labels, test_pred, zero_division=0)
    ),
    "confusion_matrix": confusion_matrix(
        test_labels, test_pred, labels=[0, 1]
    ).tolist()
}

print("\nATTENTION-BASED FUSION RESULTS")
for key, value in results.items():
    print(f"{key}: {value}")

with open(OUT_DIR / "test_metrics.json", "w") as f:
    json.dump(results, f, indent=2)

print("Saved results to:", OUT_DIR.resolve())
